# Namma Space — Self-Healing Cloud 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Round 1**

This notebook trains an authentic 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### Pre-Flight Checklist:
1. **Runtime Type**: Go to **Runtime → Change runtime type** and select **T4 GPU**.
2. **Recorded Video**: A 45–75 second MP4/MOV recorded following `docs/CAPTURE_SOP.md` (1080p landscape, locked AE/AF, dual-loop).

### Step 1: Self-Healing 3D Reconstruction Pipeline (One-Click Execute)
This cell automatically verifies GPU, clones dependencies, compiles CUDA rasterizers with `--no-build-isolation`, auto-detects your uploaded video, and executes the 30,000-iteration training.

In [ ]:
import os, sys, glob, shutil
from pathlib import Path

# 1. Verify GPU Hardware
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is NOT enabled! Please go to Runtime -> Change runtime type -> select T4 GPU.")
print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Ensure CUDA_HOME is set
if os.path.exists("/usr/local/cuda"):
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

# 2. Clone Roomsplat
if not os.path.exists("/content/roomsplat"):
    print("\n[1/5] Cloning Roomsplat...")
    !git clone https://github.com/Ahmedn1/roomsplat.git /content/roomsplat
    !pip install -e /content/roomsplat pycolmap typer rich opencv-python-headless plyfile tqdm scipy

sys.path.insert(0, "/content/roomsplat")
os.environ["PYTHONPATH"] = "/content/roomsplat"

# 3. Clone Inria Gaussian Splatting Engine
gs_dir = Path("/content/gaussian-splatting")
if not (gs_dir / "train.py").exists():
    print("\n[2/5] Cloning official Inria Gaussian Splatting engine...")
    !git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git /content/gaussian-splatting

os.environ["ROOMSPLAT_GS_DIR"] = str(gs_dir)

# 4. Link & Save Config
from roomsplat import config
config.save(gs_dir=gs_dir)
print(f"\n[3/5] Configuration saved: gs_dir={config.get_gs_dir()} (train.py: {(config.get_gs_dir() / 'train.py').exists()})")

# 5. Check & Compile Inria CUDA Rasterizers (using --no-build-isolation to access torch)
try:
    import diff_gaussian_rasterization
    import simple_knn
    print("\n[4/5] Inria CUDA rasterizers already compiled!")
except ImportError:
    print("\n[4/5] Compiling Inria CUDA rasterizers with --no-build-isolation (approx 1-2 min)...")
    !pip install --no-build-isolation /content/gaussian-splatting/submodules/diff-gaussian-rasterization
    !pip install --no-build-isolation /content/gaussian-splatting/submodules/simple-knn
    try:
        import diff_gaussian_rasterization
        print("Inria CUDA rasterizers compiled successfully!")
    except ImportError:
        print("Retrying via direct setup.py install...")
        !cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization && python3 setup.py install
        !cd /content/gaussian-splatting/submodules/simple-knn && python3 setup.py install
        import diff_gaussian_rasterization
        print("Inria CUDA rasterizers compiled successfully via setup.py!")

# 6. Auto-detect Uploaded Video
vids = [f for f in glob.glob("/content/*") if f.lower().endswith((".mp4", ".mov", ".m4v")) and not f.startswith("/content/sample_data")]
if not vids:
    from google.colab import files
    print("\nNo video detected! Please select your room walkthrough video to upload:")
    uploaded = files.upload()
    vids = list(uploaded.keys())
    if not vids:
        raise FileNotFoundError("No video was uploaded. Please re-run and upload your video.")

video_path = Path(vids[0])
print(f"\n[5/5] Target Video: {video_path} ({os.path.getsize(video_path)/(1024*1024):.2f} MB)")

# 7. Run Reconstruction Directly
from roomsplat.cli import run

print("\n=======================================================")
print("  STARTING 3D GAUSSIAN SPLATTING RECONSTRUCTION")
print("  30,000 Iterations | 1600px | Low-VRAM Mode on T4 GPU")
print("=======================================================\n")

run(
    video=video_path,
    name="Namma Space Authentic Room",
    fps=1.5,
    max_dim=1600,
    iterations=30000,
    low_vram=True,
    no_view=True
)

print("\nSUCCESS: 3D Gaussian Splatting reconstruction finished!")

### Step 2: Download `scene.splat` and `cameras.json` for Namma Space Viewer

In [ ]:
import glob
import shutil
import os
from google.colab import files

# Find output directory
splat_dirs = glob.glob("/content/*_splat")
if splat_dirs:
    out_dir = splat_dirs[0]
    print(f"Found reconstruction output at: {out_dir}")
    
    splat_file = f"{out_dir}/scene.splat"
    cameras_file = f"{out_dir}/cameras.json"
    
    if os.path.exists(splat_file) and os.path.exists(cameras_file):
        shutil.copy(splat_file, "/content/scene.splat")
        shutil.copy(cameras_file, "/content/cameras.json")
        
        # Zip together for easy download
        !zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json
        
        print("\nDownloading namma_space_assets.zip to your computer...")
        files.download("/content/namma_space_assets.zip")
    else:
        print(f"Error: scene.splat or cameras.json not found in {out_dir}!")
else:
    print("Error: Output splat directory not found! Did training complete successfully?")